In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import cross_val_score

print("Libraries imported successfully")

In [ ]:
# Load preprocessed data
data_path = Path.cwd().parent / "shared" / "output" / "production_preprocessed_data.csv"
df = pd.read_csv(data_path, encoding='utf-8-sig')

print(f"Loaded {len(df)} rows from preprocessed data")
print(f"Participants: {df['prolific_id'].nunique()}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nPhonation distribution:")
print(df['phonation'].value_counts())

In [ ]:
# ================ 참가자별 LDA 결정 경계 시각화 (확대 - 나이순 정렬) ================
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MultipleLocator
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score

print("=" * 60)
print("참가자별 LDA 결정 경계 시각화 (확대 - 나이순 정렬)")
print("=" * 60)

# 데이터 준비
lda_data_individual = df[['scaled_vot', 'normed_f0', 'phonation', 'prolific_id', 'age', 'gender']].dropna().copy()

# phonation 라벨 정규화 (데이터의 fortis를 모델/플롯에서 tense로 통일)
label_alias = {'fortis': 'tense'}
lda_data_individual['phonation_clean'] = (
    lda_data_individual['phonation']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(label_alias)
)

# 예상 라벨만 사용
valid_labels = ['lenis', 'tense', 'aspirated']
unknown_labels = sorted(set(lda_data_individual['phonation_clean']) - set(valid_labels))
if unknown_labels:
    print(f"경고: 예상하지 못한 phonation 라벨이 제외됩니다: {unknown_labels}")
lda_data_individual = lda_data_individual[lda_data_individual['phonation_clean'].isin(valid_labels)].copy()

# 시각화할 참가자 선택 (데이터가 충분한 참가자들)
valid_participants = []
for participant_id in lda_data_individual['prolific_id'].unique():
    participant_data = lda_data_individual[lda_data_individual['prolific_id'] == participant_id]
    phonation_counts = participant_data['phonation_clean'].value_counts()
    if len(phonation_counts) >= 3 and phonation_counts.min() >= 2:
        valid_participants.append(participant_id)

# 참가자를 나이순으로 정렬
participant_ages = {}
for participant_id in valid_participants:
    participant_data = lda_data_individual[lda_data_individual['prolific_id'] == participant_id]
    age = participant_data['age'].iloc[0]
    participant_ages[participant_id] = age

# 나이순으로 정렬 (오름차순)
valid_participants_sorted = sorted(valid_participants, key=lambda x: participant_ages[x])

print(f"\n시각화할 참가자 수: {len(valid_participants_sorted)}명")
print(f"나이 범위: {min(participant_ages.values()):.0f} ~ {max(participant_ages.values()):.0f}세")

# 통일된 축 범위 설정 (확대된 범위)
global_x_min = -2
global_x_max = 4
global_y_min = -3
global_y_max = 3

print(f"통일된 축 범위: X=[{global_x_min:.2f}, {global_x_max:.2f}], Y=[{global_y_min:.2f}, {global_y_max:.2f}]")

# 색상 설정
colors_dark = {'lenis': '#1f77b4', 'tense': '#ff7f0e', 'aspirated': '#2ca02c'}
colors_light = {'lenis': '#aec7e8', 'tense': '#ffbb78', 'aspirated': '#98df8a'}
label_to_num = {'lenis': 0, 'tense': 1, 'aspirated': 2}

# 5열 그리드로 모든 참가자 시각화
n_cols = 5
n_participants = len(valid_participants_sorted)
n_rows = int(np.ceil(n_participants / n_cols))

print(f"그리드 크기: {n_rows} 행 x {n_cols} 열")

# 그리드 생성
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 4))
if n_rows == 1:
    axes = axes.reshape(1, -1)
axes = axes.flatten()

for idx, participant_id in enumerate(valid_participants_sorted):
    ax = axes[idx]

    # 참가자 데이터 준비
    participant_data = lda_data_individual[lda_data_individual['prolific_id'] == participant_id].copy()

    # scaled_VOT 정규화 (참가자 내부)
    participant_data['normed_scaled_vot'] = (participant_data['scaled_vot'] - participant_data['scaled_vot'].mean()) / participant_data['scaled_vot'].std()

    X = participant_data[['normed_scaled_vot', 'normed_f0']].values
    y = participant_data['phonation_clean'].values

    try:
        # LDA 모델 학습
        lda = LinearDiscriminantAnalysis()
        lda.fit(X, y)

        # 예측 및 정확도
        y_pred = lda.predict(X)
        accuracy = accuracy_score(y, y_pred)

        # 통일된 데이터 범위 사용
        x_min, x_max = global_x_min, global_x_max
        y_min, y_max = global_y_min, global_y_max

        # Meshgrid 생성
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                             np.linspace(y_min, y_max, 200))

        # 모든 점에 대해 LDA 예측
        Z = lda.predict(np.c_[xx.ravel(), yy.ravel()])
        Z_numeric = np.array([label_to_num[label] for label in Z])
        Z_numeric = Z_numeric.reshape(xx.shape)

        # 결정 경계 영역을 색으로 채우기
        cmap = plt.matplotlib.colors.ListedColormap([colors_light['lenis'],
                                                      colors_light['tense'],
                                                      colors_light['aspirated']])
        ax.contourf(xx, yy, Z_numeric, alpha=0.3, cmap=cmap, levels=2)

        # 결정 경계선 그리기
        ax.contour(xx, yy, Z_numeric, colors='black', linewidths=1.5, linestyles='solid', levels=2)

        # 실제 데이터 포인트 scatter plot (범위 내 데이터만)
        for phonation in ['lenis', 'tense', 'aspirated']:
            mask = (y == phonation)
            if mask.sum() > 0:
                # 범위 내에 있는 데이터만 필터링
                in_range_mask = (X[mask, 0] >= x_min) & (X[mask, 0] <= x_max) & \
                               (X[mask, 1] >= y_min) & (X[mask, 1] <= y_max)
                X_filtered = X[mask][in_range_mask]
                if len(X_filtered) > 0:
                    ax.scatter(X_filtered[:, 0], X_filtered[:, 1],
                              c=colors_dark[phonation],
                              alpha=0.7, s=30, edgecolors='black', linewidth=0.5)

        # 각 클래스의 중심점 표시 (범위 내에 있을 때만)
        for phonation in ['lenis', 'tense', 'aspirated']:
            mask = (y == phonation)
            if mask.sum() > 0:
                centroid_x = X[mask, 0].mean()
                centroid_y = X[mask, 1].mean()
                # 중심점이 범위 내에 있을 때만 표시
                if (x_min <= centroid_x <= x_max) and (y_min <= centroid_y <= y_max):
                    ax.scatter(centroid_x, centroid_y,
                              c=colors_dark[phonation], marker='X', s=150,
                              edgecolors='black', linewidth=2, zorder=5)

        # 참가자 정보 추출
        participant_age = participant_data['age'].iloc[0] if 'age' in participant_data.columns else 'N/A'
        participant_gender = participant_data['gender'].iloc[0] if 'gender' in participant_data.columns else 'N/A'

        # 축 설정
        ax.set_xlabel('Normed Scaled VOT', fontsize=9)
        ax.set_ylabel('Normed F0', fontsize=9)
        ax.set_title(f'P{participant_id} ({participant_age}, {participant_gender})\nAcc: {accuracy*100:.1f}%',
                    fontsize=10, fontweight='bold')

        # 축 범위 명확히 제한
        ax.set_xlim([x_min, x_max])
        ax.set_ylim([y_min, y_max])

        # 1 z-score 단위로 그리드 설정
        ax.xaxis.set_major_locator(MultipleLocator(1))
        ax.yaxis.set_major_locator(MultipleLocator(1))
        ax.grid(True, alpha=0.3, linewidth=0.5)

        ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
        ax.axvline(x=0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
        ax.set_aspect('equal', adjustable='box')

    except Exception as e:
        # 에러 발생 시 참가자 정보 추출
        participant_age = participant_data['age'].iloc[0] if 'age' in participant_data.columns else 'N/A'
        participant_gender = participant_data['gender'].iloc[0] if 'gender' in participant_data.columns else 'N/A'
        ax.text(0.5, 0.5, f'P{participant_id} ({participant_age}, {participant_gender})\nError: {str(e)[:30]}',
               ha='center', va='center', transform=ax.transAxes, fontsize=9)
        ax.set_xlim([global_x_min, global_x_max])
        ax.set_ylim([global_y_min, global_y_max])

# 사용하지 않은 서브플롯 숨기기
for idx in range(len(valid_participants_sorted), len(axes)):
    axes[idx].axis('off')

plt.suptitle(f'Participant-Level LDA Decision Boundaries - Sorted by Age (N={len(valid_participants_sorted)})',
            fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()

# Output 폴더에 고해상도로 그림 저장
output_dir = Path.cwd() / "output"
output_dir.mkdir(exist_ok=True)
output_filename = output_dir / 'lda_decision_boundaries_by_age_highres.png'
plt.savefig(output_filename, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\n✓ 그림이 고해상도로 저장되었습니다: {output_filename}")

plt.show()

print(f"\n✓ 참가자별 LDA 결정 경계 시각화 완료 (나이순 정렬)!")
print(f"  - 총 {len(valid_participants_sorted)}명의 참가자")
print(f"  - 축 범위: X=[{global_x_min}, {global_x_max}], Y=[{global_y_min}, {global_y_max}]")
print(f"  - 정렬: 나이 오름차순")